# 6.17 · 关联规则 / Association Rule Mining

> **课程定位 / Where this fits**
> 前面的无监督找聚类/降维/异常。关联规则做**购物篮分析**: 从交易记录里挖"买了 A 的人也常买 B"的规则。经典传说: 超市发现"啤酒↔尿布"关联。它不是预测模型, 而是**发现共现模式**, 广泛用于推荐、货架摆放、捆绑销售。核心算法 **Apriori / FP-Growth**, 核心度量 **support / confidence / lift**。
> Market-basket analysis: mining "people who buy A also buy B" rules from transactions. Core algorithms Apriori/FP-Growth; core metrics support/confidence/lift.

> 💡 **面试相关 / Interview-relevant**
> - "support / confidence / lift 的定义与区别" ★★★★★
> - "为什么 confidence 高不代表有用(要看 lift)" ★★★★★
> - "Apriori 的剪枝原理(反单调性)" ★★★★
> - "Apriori vs FP-Growth" ★★★★
> - "关联 ≠ 因果" ★★★

---

## 学习目标 / Learning Objectives
1. 频繁项集 + 三大度量 support/confidence/lift。
2. Apriori 的反单调剪枝; FP-Growth 的提速。
3. 用 mlxtend 挖规则并解读。
4. 为何要看 lift(confidence 的陷阱)。

## 目录 / TOC
1. [频繁项集与三度量 ⭐](#1)
2. [🛒 数据: 购物篮(合成)](#2)
3. [Apriori 挖频繁项集 + 规则 ⭐](#3)
4. [lift 的重要性 ⭐](#4)
5. [FP-Growth + 小结](#5)


<a id="1"></a>
## 1. 频繁项集与三度量 ⭐ / Frequent Itemsets & Metrics

**项集(itemset)**: 一次交易里一起出现的商品集合。**频繁项集**: 出现频率 ≥ 阈值的项集。规则形如 $A\Rightarrow B$($A,B$ 是项集)。三大度量:

- **支持度 support**$(A) = \Pr(A)$ = 含 A 的交易占比。衡量**普遍性**(太低的规则不值得关注)。
- **置信度 confidence**$(A\Rightarrow B)=\Pr(B\mid A)=\frac{\text{support}(A\cup B)}{\text{support}(A)}$。买了 A 的人里有多大比例也买 B。
- **提升度 lift**$(A\Rightarrow B)=\frac{\text{confidence}}{\Pr(B)}=\frac{\Pr(A\cap B)}{\Pr(A)\Pr(B)}$。
  - lift > 1: A 和 B **正相关**(A 促进 B), 规则有价值。
  - lift = 1: **独立**(没关系)。
  - lift < 1: 负相关。

**关键**(面试核心): **高 confidence 可能是假象**——若 B 本身就极普遍(如"购物袋"), 任何 A⇒B 的 confidence 都高, 但其实没关联。**lift 才剔除了 B 的基础流行度**, 衡量真正的关联强度。


<a id="2"></a>
## 2. 数据: 购物篮(合成) / Synthetic Market Basket

合成一个杂货店交易集: 每行一次购物。**埋入真实关联**——比如"面包+黄油""啤酒+尿布""咖啡+牛奶"常一起买, 另放一个**高频但与谁都无关**的商品(购物袋)来演示 lift 的必要性。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

def make_baskets(n=2000, seed=0):
    rng = np.random.default_rng(seed)
    baskets = []
    for _ in range(n):
        items = set()
        if rng.random() < 0.5: items |= {"面包","黄油"}          # 面包→黄油关联
        if rng.random() < 0.3: items |= {"啤酒","尿布"}          # 经典关联
        if rng.random() < 0.4: items |= {"咖啡","牛奶"}          # 关联
        for it,p in [("鸡蛋",0.3),("苹果",0.25),("薯片",0.2),("可乐",0.2)]:
            if rng.random() < p: items.add(it)
        if rng.random() < 0.85: items.add("购物袋")               # 高频但无关联(干扰项)
        if items: baskets.append(list(items))
    return baskets

baskets = make_baskets()
print(f"购物篮: {len(baskets)} 次交易")
for b in baskets[:5]: print("  ", b)
print(f"\n平均每篮 {np.mean([len(b) for b in baskets]):.1f} 件商品")


<a id="3"></a>
## 3. Apriori 挖频繁项集 + 规则 ⭐ / Apriori

**Apriori 核心剪枝(反单调性)**: 如果一个项集不频繁, 它的**所有超集也必不频繁**(支持度只会更低)。所以从单品开始逐层扩展, 一旦某项集低于 min_support 就**剪掉**, 不再考虑它的超集——大幅减少搜索。用 `mlxtend` 实现。


In [ ]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

te = TransactionEncoder()
arr = te.fit_transform(baskets)
df = pd.DataFrame(arr, columns=te.columns_)   # one-hot 交易矩阵
print(f"交易矩阵(one-hot): {df.shape}")

freq = apriori(df, min_support=0.05, use_colnames=True)
freq["len"] = freq["itemsets"].apply(len)
print(f"\n频繁项集(support≥0.05): {len(freq)} 个")
print(freq[freq.len>=2].sort_values("support", ascending=False).head(8).to_string(index=False))

rules = association_rules(freq, metric="confidence", min_threshold=0.5)
rules = rules[rules["lift"] > 1.0].sort_values("lift", ascending=False)
print("\nTop 关联规则(按 lift):")
show = rules[["antecedents","consequents","support","confidence","lift"]].head(8).copy()
for c in ["antecedents","consequents"]:
    show[c] = show[c].apply(lambda s: ",".join(s))
print(show.round(3).to_string(index=False))


<a id="4"></a>
## 4. lift 的重要性 ⭐ / Why Lift Matters

来看"购物袋"陷阱: 它出现在 85% 的交易里。任何"X ⇒ 购物袋"的规则 **confidence 都很高**(~0.85), 看着像强规则, 但 **lift ≈ 1**——因为购物袋跟谁都一起出现, 没有真正关联。这就是为什么**只看 confidence 会被高频项骗, 必须看 lift**。


In [ ]:
# 比较: 含"购物袋"的规则 vs 真实关联规则 / bag-trap vs real association
all_rules = association_rules(freq, metric="confidence", min_threshold=0.3)
# 只看 单品⇒单品 的规则, 便于清晰解读
all_rules["ant"] = all_rules["antecedents"].apply(lambda s: ",".join(s))
all_rules["con"] = all_rules["consequents"].apply(lambda s: ",".join(s))
one2one = all_rules[(all_rules["antecedents"].apply(len)==1) & (all_rules["consequents"].apply(len)==1)]

bag = one2one[one2one["con"]=="购物袋"].sort_values("confidence", ascending=False)
print("'X ⇒ 购物袋' 规则: confidence 高(~0.85)但 lift≈1(假关联, 购物袋跟谁都一起出现)")
print(bag[["ant","con","confidence","lift"]].head(4).round(3).to_string(index=False))

real = one2one[(one2one["con"]!="购物袋") & (one2one["ant"]!="购物袋")].sort_values("lift", ascending=False)
print("\n真实关联规则(单品⇒单品, lift 明显 >1): 啤酒↔尿布 / 面包↔黄油 / 咖啡↔牛奶")
print(real[["ant","con","confidence","lift"]].head(6).round(3).to_string(index=False))
print("\n→ confidence 高 ≠ 有用; lift 剔除了 consequent 的基础流行度, 才是真关联强度")


<a id="5"></a>
## 5. FP-Growth + 小结 / FP-Growth & Summary

**FP-Growth**: Apriori 要反复扫数据库、生成大量候选项集, 慢。FP-Growth 把数据压成一棵 **FP-tree**, 只扫两遍数据库、不显式生成候选, 通常**快得多**。mlxtend 里接口与 apriori 一致, 可直接替换。


In [ ]:
from mlxtend.frequent_patterns import fpgrowth
import time

t = time.perf_counter(); fa = apriori(df, min_support=0.05, use_colnames=True); t_ap = time.perf_counter()-t
t = time.perf_counter(); ff = fpgrowth(df, min_support=0.05, use_colnames=True); t_fp = time.perf_counter()-t
print(f"Apriori : {len(fa)} 频繁项集, {t_ap*1000:.1f} ms")
print(f"FP-Growth: {len(ff)} 频繁项集, {t_fp*1000:.1f} ms (结果相同, 大数据上通常更快)")
print("两者找到相同的频繁项集; FP-Growth 用 FP-tree 避免候选生成, 大规模交易上更快")


### 小结 / Summary

```
关联规则: 购物篮分析, 挖 A⇒B 共现模式(非预测, 是发现)
support=P(A) 普遍性; confidence=P(B|A); lift=P(A∩B)/(P(A)P(B)) 关联强度
lift>1 正相关(有用) =1 独立 <1 负相关; ⚠️高confidence可能是B太普遍(看lift!)
Apriori: 反单调剪枝(不频繁项集的超集必不频繁), 逐层扩展
FP-Growth: FP-tree 压缩, 不生成候选, 大数据更快
关联 ≠ 因果(只是共现)
```

### 💡 面试速查
1. **support(普遍)/ confidence(P(B|A))/ lift(关联强度)**
2. **lift>1 才有价值**; 高 confidence 可能因 B 本身高频(陷阱) → 看 lift
3. **Apriori 反单调剪枝**: 子集不频繁→超集必不频繁
4. **FP-Growth**: FP-tree 免候选生成, 比 Apriori 快
5. **关联≠因果**, 只是统计共现; 用于推荐/捆绑/货架

### 下一节
**6.18 NMF**——非负矩阵分解: 把非负数据(如词频、像素、评分)分解成非负的"部件"之和, 天然可解释(主题/部件), 是降维与主题建模的另一利器, 为 Part 6 收尾。
